# LoRA SFT for text-to-SQL — Qwen2.5-1.5B on Spider

This notebook produces the **only** numbers allowed to appear in the project README.
It fine-tunes LoRA adapters on a frozen 4-bit Qwen2.5-1.5B-Instruct, then evaluates
three systems on Spider's held-out dev set and writes `metrics.json`.

It contains **no metric code**. Everything is imported from the `text2sql_lora`
package, which is unit-tested on CPU and whose harness has already been verified to
score Spider's gold queries against themselves at 1.0000. This notebook is
orchestration only.

## Before you hit Run All — sidebar settings

1. **Settings → Accelerator → GPU T4 x2** (P100 also fine; only one GPU is used).
2. **Settings → Internet → On.** Required to download the base model and Spider.

## Getting this repo onto Kaggle

**Route A (preferred):** set `REPO_URL` in the next cell.
If the repo is **private**, add a `GITHUB_TOKEN` secret (*Add-ons → Secrets*, with
`repo` scope) and load it before the install cell:

```python
from kaggle_secrets import UserSecretsClient
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
```

The variable must be named exactly `GITHUB_TOKEN` — the install cell looks it up by
that name, injects it into the clone URL without printing it, and strips it back out
of `.git/config` immediately afterwards.

**Route B (fallback):** zip the repo, upload it as a Kaggle Dataset
(*Datasets → New Dataset → Upload*), and attach it via *Add Input*. The install cell
finds it automatically by looking for `pyproject.toml`. Exclude `.venv/` and `data/`.

## Expected runtime

Roughly 1.5–2.5 h for training plus 30–60 min for the three evaluation passes, inside
Kaggle's 9 h session limit. The smoke cell runs first so a broken setup costs two
minutes rather than two hours.

In [ ]:
# Pin to ONE GPU, before torch is imported.
#
# Kaggle's "T4 x2" exposes two devices, and HuggingFace Trainer silently wraps the model
# in nn.DataParallel whenever it sees more than one. DataParallel replicates the model
# across devices each step, and a bitsandbytes 4-bit layer does NOT survive replication --
# the quantization state is left behind, and training dies with
# "CUDA error: an illegal memory access was encountered" from inside gemm_4bit.
#
# This run is sized for a single <=8 GB GPU by design, so the second device was never
# going to be used. Setting this before `import torch` is the reliable place to do it.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Route A: your GitHub repo. If it is PRIVATE, run the GITHUB_TOKEN secret cell first
# (Add-ons -> Secrets), and the install cell below will authenticate the clone.
REPO_URL = "https://github.com/shashi05shankar/Text-to-SQL-LoRA-Supervised-Fine-Tuning.git"

import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("visible GPUs:", torch.cuda.device_count(), "(must be 1)")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"gpu: {name} | compute capability {major}.{minor} | {total_gb:.1f} GB")
    # bfloat16 needs 8.0+. The T4 is 7.5, so the run uses fp16 -- this is not a fallback
    # that silently degrades quality, it is the only 16-bit format Turing supports.
    print("dtype:", "bfloat16" if major >= 8 else "float16 (Turing: no bf16)")
    if torch.cuda.device_count() > 1:
        print(
            "WARNING: more than one GPU is still visible. torch was already imported "
            "before CUDA_VISIBLE_DEVICES was set -- use Run -> Restart & Clear Cell "
            "Outputs, then run this cell FIRST."
        )
else:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")

In [ ]:
# Install the project. Route A (git clone) is tried first; if it fails, fall back to
# Route B, a copy attached as a Kaggle Dataset.
#
# PRIVATE REPO: define GITHUB_TOKEN in an earlier cell (Add-ons -> Secrets) and it is
# injected into the clone URL here. Two precautions, because a leaked PAT is a real
# problem and Kaggle notebooks are easy to share publicly:
#   * the authenticated URL is never printed, and never assigned to a global;
#   * the remote is rewritten to the token-free URL immediately after cloning, so the
#     token does not linger in .git/config inside /kaggle/working.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/text2sql-lora-sft")


def authenticated(url: str) -> str:
    token = globals().get("GITHUB_TOKEN")
    if not token or not url.startswith("https://github.com/"):
        return url
    return url.replace("https://", f"https://x-access-token:{token}@", 1)


def find_attached_repo():
    """Locate this repo among the attached datasets, by its pyproject."""
    for candidate in Path("/kaggle/input").rglob("pyproject.toml"):
        try:
            if "text2sql-lora" in candidate.read_text(encoding="utf-8"):
                return candidate.parent
        except Exception:
            continue
    return None


if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

cloned = False
try:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", authenticated(REPO_URL), str(REPO_DIR)],
        capture_output=True, timeout=180, text=True,
    )
    if result.returncode != 0:
        # Scrub the token out of the error before showing it.
        message = result.stderr.replace(str(globals().get("GITHUB_TOKEN", "\0")), "***")
        raise RuntimeError(message.strip()[:400])
    cloned = True
    # Drop the credential from .git/config.
    subprocess.run(
        ["git", "remote", "set-url", "origin", REPO_URL],
        cwd=str(REPO_DIR), capture_output=True,
    )
    print(f"Route A: cloned {REPO_URL}")
except Exception as exc:
    print(f"Route A failed: {exc}")
    print("Trying Route B (attached dataset)...")

if not cloned:
    attached = find_attached_repo()
    if attached is None:
        raise SystemExit(
            "Repo not found. Either make REPO_URL reachable (public, or private with a "
            "valid GITHUB_TOKEN secret that has 'repo' scope), or upload the repo as a "
            "Kaggle Dataset and attach it via Add Input."
        )
    # /kaggle/input is read-only; copy so the editable install can write egg-info.
    shutil.copytree(attached, REPO_DIR)
    print(f"Route B: copied {attached}")

sys.path.insert(0, str(REPO_DIR / "src"))

# Fail now, with a clear message, rather than three cells later on an ImportError.
if not (REPO_DIR / "src" / "text2sql_lora" / "train" / "sft.py").exists():
    raise SystemExit(f"repo at {REPO_DIR} is missing src/text2sql_lora/train/sft.py")
print("repo ready at", REPO_DIR)

In [ ]:
# Kaggle ships transformers/torch already. Only the pieces it may lack are installed,
# and versions are left alone otherwise -- upgrading torch on Kaggle usually breaks CUDA.
!pip install -q peft bitsandbytes accelerate

import importlib

import peft
import peft.import_utils
import transformers

print("transformers", transformers.__version__, "| peft", peft.__version__)
import bitsandbytes

print("bitsandbytes", bitsandbytes.__version__)

# Kaggle preinstalls torchao 0.10.0. PEFT's LoRA dispatcher calls is_torchao_available(),
# which RAISES on a version below its minimum rather than skipping -- which aborts adapter
# injection (ImportError deep inside get_peft_model). We never use torchao (optional
# quantization backend, unrelated to our NF4/bitsandbytes path), so remove it and let the
# check return False cleanly.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
importlib.invalidate_caches()
for _mod in list(sys.modules):
    if _mod == "torchao" or _mod.startswith("torchao."):
        del sys.modules[_mod]
if hasattr(peft.import_utils.is_torchao_available, "cache_clear"):
    peft.import_utils.is_torchao_available.cache_clear()
print("torchao available to peft:", peft.import_utils.is_torchao_available())  # expect False

In [ ]:
# Download Spider: the question/SQL parquet AND the ~206 MB database archive. Execution
# accuracy is impossible without the databases, so both are required.
os.environ["T2S_DATA_DIR"] = "/kaggle/working/data"
os.environ["T2S_ARTIFACTS_DIR"] = "/kaggle/working/artifacts"

subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "download_data.py")],
    check=True, cwd=str(REPO_DIR),
)

In [ ]:
# Verify the harness BEFORE training. Scoring gold against itself must come out ~1.0.
# If it does not, the metric is broken and nothing trained afterwards would mean anything
# -- far better to learn that now than after an hour on the GPU.
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "check_harness.py"), "--limit", "200"],
    check=True, cwd=str(REPO_DIR),
)

In [ ]:
import text2sql_lora.config as t2s_config
import text2sql_lora.train.sft as sft

importlib.reload(t2s_config)
importlib.reload(sft)

PATHS = t2s_config.Paths()
DATA_CONFIG = t2s_config.DataConfig()
LORA_CONFIG = t2s_config.LoraConfig()
TRAIN_CONFIG = t2s_config.TrainConfig()

print("data     :", PATHS.data_dir)
print("artifacts:", PATHS.artifacts_dir)
print("base     :", DATA_CONFIG.base_model)
print(f"lora     : r={LORA_CONFIG.r} alpha={LORA_CONFIG.alpha} -> {list(LORA_CONFIG.target_modules)}")
print(
    f"train    : {TRAIN_CONFIG.epochs} epochs, batch "
    f"{TRAIN_CONFIG.per_device_batch_size}x{TRAIN_CONFIG.gradient_accumulation_steps}"
    f" = {TRAIN_CONFIG.per_device_batch_size * TRAIN_CONFIG.gradient_accumulation_steps}"
    f", seq {DATA_CONFIG.max_seq_len}, lr {TRAIN_CONFIG.learning_rate}"
)

In [ ]:
# SMOKE RUN: 5 optimizer steps, 8 dev examples. Its numbers are meaningless by
# construction -- it exists to surface OOM, dtype and shape errors in ~2 minutes.
# If this raises, stop and fix it. Do not proceed to the real run.
#
# Path(), not a bare string: `Paths` now coerces strings, but being explicit here keeps
# the cell working against an older clone of the repo too.
from pathlib import Path

smoke = sft.run_smoke(
    device="cuda",
    paths=t2s_config.Paths(artifacts_dir=Path("/kaggle/working/smoke")),
)
print("\nsmoke OK -- pipeline runs. Ignore the accuracy above.")

In [ ]:
# Free the smoke model before the real run, or the two compete for VRAM.
import gc

del smoke
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# THE REAL RUN. Trains LoRA on all 6,300-ish training examples (7,000 minus the
# database-disjoint monitoring split), then evaluates three systems on all 1,034 dev
# questions: base zero-shot, base 3-shot, and LoRA-tuned zero-shot.
#
# Dev is touched here and nowhere else in the project.
payload = sft.run(
    device="cuda",
    paths=PATHS,
    data_config=DATA_CONFIG,
    lora_config=LORA_CONFIG,
    train_config=TRAIN_CONFIG,
    generation_batch_size=16,
)

In [ ]:
# The headline. These are the numbers the README is allowed to quote -- nothing else is.
import pandas as pd

results = pd.DataFrame(payload["results"]).T[
    ["n", "execution_accuracy", "exact_match", "invalid_sql_rate"]
]
display(results.round(4))

print("\nheadline:")
for key, value in payload["headline"].items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

print("\nby hardness (our own component-count heuristic, not Yale's definition):")
display(pd.DataFrame(payload["by_hardness"]).pivot(
    index="hardness", columns="system", values="execution_accuracy"
).round(4))

manifest = payload["manifest"]
print(
    f"\ntrainable {manifest['trainable_params']:,} / {manifest['total_params']:,} "
    f"({manifest['trainable_percent']:.3f}%) | wall clock "
    f"{manifest['wall_clock_seconds'] / 60:.1f} min | {manifest.get('gpu', '?')}"
)

In [ ]:
# Eyeball real predictions before trusting the table. Cases where fine-tuning changed the
# verdict are the interesting ones -- and any case where it made things WORSE belongs in
# the README too.
import json

rows = [json.loads(line) for line in open(PATHS.predictions_jsonl(), encoding="utf-8")]
fixed = [r for r in rows
         if r["lora_zero_shot"]["execution_correct"] and not r["base_zero_shot"]["execution_correct"]]
broke = [r for r in rows
         if r["base_zero_shot"]["execution_correct"] and not r["lora_zero_shot"]["execution_correct"]]

print(f"fine-tuning FIXED {len(fixed)} and BROKE {len(broke)} of {len(rows)}\n")
for row in fixed[:3]:
    print(f"[{row['db_id']}] {row['question']}")
    print(f"  gold : {row['gold']}")
    print(f"  base : {row['base_zero_shot']['prediction'][:160]}")
    print(f"  tuned: {row['lora_zero_shot']['prediction'][:160]}\n")

In [ ]:
# Plot the training curve from the recorded log history (also saved inside metrics.json,
# so the chart can be regenerated locally without rerunning anything).
import matplotlib.pyplot as plt

history = payload["training"]["log_history"]
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(8, 4))
if train_points:
    ax.plot(*zip(*train_points), label="train loss", linewidth=1)
if eval_points:
    ax.plot(*zip(*eval_points), label="val loss (db-disjoint split)", marker="o")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("LoRA SFT — Qwen2.5-1.5B-Instruct on Spider")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PATHS.loss_curve_png(), dpi=150)
plt.show()

In [ ]:
# Package the artifacts for download as ONE file.
#
# Download `artifacts.zip` from the Output panel on the right, then unzip it into the
# repo's `artifacts/` directory locally:
#
#     artifacts/adapter/            <- LoRA weights + tokenizer
#     artifacts/metrics.json        <- every number the README quotes
#     artifacts/predictions.jsonl   <- per-question audit trail, drives the demo
#     artifacts/loss_curve.png
#
# The base model is NOT in here and does not need to be -- it is downloaded from the Hub
# on first local use. That is the point of LoRA: the trained part is tens of MB.
shutil.rmtree(PATHS.artifacts_dir / "training", ignore_errors=True)  # optimizer junk

archive = shutil.make_archive("/kaggle/working/artifacts", "zip", PATHS.artifacts_dir)
size_mb = Path(archive).stat().st_size / 1e6
print(f"{archive}  ({size_mb:.1f} MB)")
for item in sorted(PATHS.artifacts_dir.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(PATHS.artifacts_dir)}  {item.stat().st_size / 1e6:.2f} MB")